# LiteraryWorksMetaDataUploadBot

In [ ]:
import pandas as pd
import re
from datetime import datetime
import wikibaseintegrator
from wikibaseintegrator import WikibaseIntegrator, wbi_helpers, wbi_login, datatypes
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.entities import ItemEntity
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path
from typing import Optional, Union
from dataclasses import dataclass, field
from __future__ import annotations
from enum import Enum
import numpy as np
from time import sleep

collections_dir = Path("../wikidata/metadata_collections/")

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)



BOTNAME = authorization['BOTNAME']
MY_USERNAME = authorization['MY_USERNAME']
    

log_file = str(Path('logs/Mass_Upload.log'))
logging.basicConfig(filename=log_file, force=True,
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)

logger = logging.getLogger('Make-Items')
logger.debug('Start logging')


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{authorization['BOT_USERNAME']})'
#wbi_config['MEDIAWIKI_API_URL'] = 'https://test.wikidata.org/w/api.php'
wbi_config['MEDIAWIKI_API_URL'] = 'https://www.wikidata.org/w/api.php'


PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'edition_of': 'P629',
         'based_on' : 'P144',
         'language':'P407',
         'publication_date':'P577',
         'work_available_at_URL':'P953',
         'has_edition_or_translation' : 'P747',
         'project_gb_ebook_id' : 'P2034',
         'sex_or_gender' :'P21',
         'family_name' : 'P734',
         'given_name' : 'P735',
         'date_of_birth' : 'P569',
         'date_of_death' : 'P570'
         }
ENTITIES = {
   'literary_work':'Q7725634', 
   'edition' : 'Q3331189',
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

### The Bot

In [ ]:
#
# THE UPLOAD SCRIPT / BOT
#

def date_tag_precise():
    return datetime.today().strftime("%Y-%m-%dT%H:%M:%S")

def format_year(year : int):
    return datetime(year, 1, 1).strftime("+%Y-%m-%dT%H:%M:%SZ")

class Author():
    def __init__(self, item = None, qid=None, name = None, works = None, editions = None):
        self._item = item
        self._qid = qid
        self._name = name
        self._works = works
        self._editions = editions
    
    @classmethod
    def from_item(self, item: wikibaseintegrator.entities.item.ItemEntity):
        return Author(item=item)
    
    def get_name(self, language='mul'):
        if self._item:
            if self._item.labels.get(language):
                return self._item.labels.get(language).value
            if self._item.labels.get('mul'):
                return self._item.labels.get('mul').value
            if self._name:
                return self._name
            else:
                if self._quid:
                    raise Exception(f'No name available for {self._qid}')
                if self.item:
                    raise Exception(f'No name available for {self._item}')
                raise Exception(f'No name available for {self}')
        else:
            return self.name
        

class Bot():
    def __init__(this, wbi, login_instance, EDIT_SUMMARY, logger, save_dir = Path(".")):
        this.wbi = wbi
        this.login_instance = login_instance
        this.EDIT_SUMMARY = EDIT_SUMMARY
        this.logger = logger
        this.save_dir = save_dir

    def make_work(this, author_qid : str, author : Author, title : str, 
                year : int = None, url : Optional[str] = None): 
        language = ENTITIES['German']

        formatted_year = format_year(year)

        new_work = this.wbi.item.new()

        new_work.labels.set('de', title)
        # Set a default label too
        new_work.labels.set('mul', title)
        new_work.descriptions.set('en', f'Literary work of fiction by {author.get_name('en')}')
        new_work.descriptions.set('de', f'Fiktionales literarisches Werk von {author.get_name('de')}')
        
        new_work.claims.add([
            datatypes.Item(value=ENTITIES['literary_work'], prop_nr=PROPS['instance_of']), 
            datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
            #
            # TODO: DEAL WITH SEVERAL AUTHORS (we currently make sure only one author works BEFORE we start upload)
            #
            datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
            datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
            datatypes.Item(value=language, prop_nr=PROPS['language'])
                        ])
        if url:
            new_work.claims.add([
                datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                            ])
        return new_work



    def make_edition(this, author_qid : str, title : str, author : Author, year :int, 
                    work_qid : Optional[str], url : Optional[str] = None): 
        language = ENTITIES['German']

        new_edition = this.wbi.item.new()
        new_edition.labels.set('de', f'{title} (Erstausgabe von {str(year)})')
        # Set a default label too
        new_edition.labels.set('mul', f'{title} (first edition, {str(year)})')

        new_edition.descriptions.set('en', 
                            f'{str(year)} edition of the literary work of fiction by {author.get_name('en')}')
        new_edition.descriptions.set('de', 
                            f'Ausgabe von {str(year)} des fiktionalen literarischen Werks von {author.get_name('de')}')

        formatted_year = format_year(year)
        
        new_edition.claims.add([
            datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
            datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
            datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
            datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
            datatypes.Item(value=language, prop_nr=PROPS['language'])
                                ])   

        if work_qid: 
            new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                                ])
            
        if url:
            new_edition.claims.add([
                datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                            ])

        return new_edition


    def make_gutenberg_edition(this, author_qid : str, title : str, author : Author, source : str,
                            work_qid : Optional[str] = None, based_on_edition_qid : Optional[str] = None, 
                            url : Optional[str] = None, year : Optional[int]=None, pg_id : Optional[str] = None): 

        language = ENTITIES['German']

        new_edition = this.wbi.item.new()

        if source == 'PG-DE':
            gb_edition_descr = {'en': 'Projekt Gutenberg-DE edition', 
                            'de': 'Projekt Gutenberg-DE Edition'}
        elif source == 'PG-US':
            gb_edition_descr = {'en': 'Project Gutenberg edition', 
                            'de': 'Project Gutenberg Edition'}
        else:
            raise Exception(f"source must be one of 'PG-DE' or 'PG-US'")

        new_edition.labels.set('de', title + f' ({gb_edition_descr['de']})')
        # Set a default label too
        new_edition.labels.set('mul', title + f' ({gb_edition_descr['en']})')

        new_edition.descriptions.set('en', 
                    f'{gb_edition_descr['en']} of the literary work of fiction by {author.get_name('de')}')
        new_edition.descriptions.set('de', 
                    f'{gb_edition_descr['de']} des fiktionalen literarischen Werks von {author.get_name('de')}')

        new_edition.claims.add([
            datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
            datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of']),
            datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
            datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
            # We don't have the year when PG version was published (yet?), so leave it out
            #datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
            datatypes.Item(value=language, prop_nr=PROPS['language'])
                                ])

        if work_qid: 
            new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                                ])
            
        if based_on_edition_qid: 
            new_edition.claims.add([ datatypes.Item(value=based_on_edition_qid, prop_nr=PROPS['based_on'])
                                ])   
        if url:
            new_edition.claims.add([
                datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                            ])
            
        if year:
            formatted_year = format_year(year)
            new_edition.claims.add([
                datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
                            ])
                    # already made sure its a string in the function header, but let's be robust
        if pg_id and (source == 'PG-US'):
                new_edition.claims.add([
                    datatypes.ExternalID(value=pg_id, prop_nr=PROPS['project_gb_ebook_id']),
                            ])
            
        return new_edition


    def add_editions_to_work(this, work_qid : str, edition_qids : list[str], edit_summary = None):
        work = this.wbi.item.get(entity_id=work_qid)
        claims_to_add = [datatypes.Item(value=edition_qid, prop_nr=PROPS['has_edition_or_translation']) 
                        for edition_qid in edition_qids]
        work.claims.add(claims_to_add)
        sleep(5)
        work.write(summary = edit_summary)
        


    def upload(this, data_for_upload, metadata, dry_run = True):
        # BOTH TODOs should be fixed now:
        # TODO: There is a warning here, meaning we forget to set dtype on column first:
        # /var/folders/ck/dw0r0d9x6v9d5y4pgvkx70x80000gn/T/ipykernel_38899/257793778.py:70: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Q136801459' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
        #
        # formatted_year isn't used


        try:
            for work in data_for_upload.itertuples():
                for col in ['author_qid', 'author', 'title', 'year', 'source']:
                    if pd.isna(getattr(work, col)) or (getattr(work, col) == 'nan'):
                        msg = f'Has NaN in metadata, skipping: {i} col: {col}'
                        print(msg)
                        this.logger.warning(msg)
                        continue
                if pd.isna(work.url): #or (work.url == 'nan'):
                    url = None
                else: 
                    url = work.url.strip()

                if pd.isna(work.gutenberg_id): # or (work.source == 'nan'):
                    pg_id = None
                else: 
                    pg_id = work.gutenberg_id.strip()

                author_qid = work.author_qid.strip()
                author_entity = this.wbi.item.get(entity_id=author_qid)
                author = Author.from_item(author_entity)
                title = work.title.strip()
                source = work.source.strip()

                this.logger.info(
                    f'next upload: {work[0]}, {author_qid}, {work.author}, {title}, {work.year}, {source}, {url}, {pg_id}'
                    )
                new_work = this.make_work(author_qid = author_qid, author=author, title = title, 
                                year = work.year, url=url)
                
                if dry_run:
                    this.logger.info(f'(dry run) would write work item: {new_work.get_json()}')
                else:
                    sleep(5)
                    new_work.write(summary=this.EDIT_SUMMARY)
                    this.logger.info(f'written work item: {new_work.get_json()}')
                    
                work_qid = new_work.id
                this.logger.info(f'work item id: {work_qid}')

                metadata.at[work[0], 'work_qid_by'] = work_qid
                metadata.at[work[0], 'work_qid_date'] = date_tag_precise()

                new_edition = this.make_edition(author_qid = author_qid, year = work.year,
                                    author = author,
                                    title = title, 
                                    work_qid = work_qid)
                
                
                if dry_run:
                    this.logger.info(f'(dry run) would write edition item: {new_edition.get_json()}')
                else:
                    sleep(5)
                    new_edition.write(summary=this.EDIT_SUMMARY)
                    this.logger.info(f'written edition item: {new_edition.get_json()}')

                
                edition_qid = new_edition.id
                this.logger.info(f'edition item id: {edition_qid}')

                metadata.at[work[0], 'ed1_qid_by'] = edition_qid
                metadata.at[work[0], 'ed1_qid_date'] = date_tag_precise()

                new_pg_edition = this.make_gutenberg_edition(author_qid = author_qid, 
                                    title = title, 
                                    author = author, 
                                    work_qid= work_qid,
                                    #based_on_edition_qid = edition_qid, 
                                    pg_id=pg_id, 
                                    source=source,
                                    url=url)
                if dry_run:
                    this.logger.info(f'(dry run) would write PG ed item: {new_pg_edition.get_json()}')
                else:
                    sleep(5)
                    new_pg_edition.write(summary=this.EDIT_SUMMARY)
                    this.logger.info(f'written PG ed item: {new_pg_edition.get_json()}')
                

                pg_edition_qid = new_pg_edition.id
                this.logger.info(f'PG edition item id: {pg_edition_qid}')

                if source == 'PG-DE':
                    metadata.at[work[0], 'pgde_qid_by'] = work_qid
                    metadata.at[work[0], 'pgde_qid_date'] = date_tag_precise()
                elif source == 'PG-US':
                    metadata.at[work[0], 'pgus_qid_by'] = work_qid
                    metadata.at[work[0], 'pgus_qid_date'] = date_tag_precise()

                if not dry_run:
                    this.add_editions_to_work(work_qid=work_qid, edition_qids=[edition_qid, 
                                                                               pg_edition_qid], 
                                                                               edit_summary=this.EDIT_SUMMARY)
                    this.logger.info(f'added editions {edition_qid}, {pg_edition_qid} to {work_qid}')
                else: 
                    this.logger.info(f'(dry run) would add editions {edition_qid}, {pg_edition_qid} to {work_qid}')

        except Exception as e:
            this.logger.error(f'Error during upload of metadata for index:{work[0]}, filename: {work.filename}')
            this.logger.error(e)

        finally:
            if not dry_run:
                metadata.to_csv(Path(this.save_dir,f'de_fiction_metadata_{date_tag_precise()}.csv'))

# Preparing the data for upload

We make sure no parts of works or collections are uploaded for now. (These special cases will be dealt with in a future version.)
We also currently do not upload works with more than one author. We also currently only upload authors which do not have any works at all on wikidata. In the future, we will find ways to decide when an existing work matches a work in our metadata set and how to reconcile the two (perhaps this will be done via a different method such as OpenRefine).

In [ ]:
#metadata = pd.read_csv(Path(collections_dir, 'de_fiction_metadata_2025-11-14T18.csv'), 
#                       index_col=0, dtype={'gutenberg_id':str})
metadata = pd.read_csv(Path(collections_dir, 'de_fiction_metadata_2026-01-05T18.csv'), 
                       index_col=0, dtype={'gutenberg_id':str})
metadata

In [ ]:
author_qids = set(metadata['author_qid'].dropna())
print(len(author_qids))
author_qids

In [ ]:
# Executes a SPARQL query based on list of qids

def query_works(author_qids : list[str]):
    query = """SELECT ?author_qid ?author_qidLabel ?work_qid ?work_qidLabel ?title ?title_de ?work_alt ?publication_date ?language
        WHERE
        {
        VALUES ?author_qid { wd:""" + " wd:".join(author_qids) + """}
        ?work_qid wdt:P31 wd:Q7725634 .
        ?work_qid wdt:P50 ?author_qid .
        OPTIONAL {
            ?work_label rdfs:label ?work_qid .
            }
        OPTIONAL {
            ?work_qid skos:altLabel ?work_alt FILTER (lang(?work_alt) = "de") .
            }
        OPTIONAL {
            ?work_qid wdt:P1476 ?title . 
            }
        OPTIONAL {
            ?work_qid wdt:P1476 ?title_de FILTER (lang(?title_de) = "de") . 
            }
        OPTIONAL {
            ?work_qid wdt:P577 ?publication_date .
            }
            OPTIONAL {
            ?work_qid wdt:P407 ?language .
            }
        SERVICE wikibase:label { bd:serviceParam wikibase:language "de,mul,en". }
        }
        ORDER BY desc(?author_qidLabel) desc(?title)"""
    return wbi_helpers.execute_sparql_query(query, 
                                            user_agent=wbi_config['USER_AGENT'])

# Here's how to use it:
works_raw = query_works(['Q70989'])
works_raw['head']

In [ ]:
def qid_from_url(url : str):
    return re.search(r'Q\d+$',url).group()

def chunker_alt(seq, size):
    return [seq[pos:pos + size] for pos in range(0, len(seq), size)]

def chunk_lengths(total_sents, num_chunks):
  base_chunk_size, remainder = divmod(total_sents, num_chunks)
  return [base_chunk_size + 1] * remainder + [base_chunk_size] * (num_chunks - remainder)

def chunker(seq, size = None, num_chunks = None, start_at = None, stop_before = None, ind = None, return_ind = False):
  if num_chunks == None and size == None:
    raise RuntimeError("One of num_chunks and size must be given!")

  if (num_chunks and (len(seq) < num_chunks)) or (size and (len(seq) < size)):
     raise RuntimeError("sequence is too short to be chunked in this way")
  #make chunks of fixed size if possible 
  if (size != None) and (num_chunks == None):
    all_chunks = [seq[pos:pos + size] for pos in range(0, len(seq), size)]
  
  #guarantee certain number of chunks
  elif (size == None) and (num_chunks != None):
    lengths = chunk_lengths(len(seq), num_chunks)
    sums = [sum(lengths[0:i]) for i in range(0, num_chunks + 1)]
    chunk_borders = zip(sums[:-1], sums[1:])
    all_chunks = [seq[start:end] for start, end in chunk_borders]
  else:
    raise RuntimeError("chunker should not be called with both size and num_chunks args")
  
  # If only part requested...
  if start_at == None:
    start_at = 0
  if stop_before == None:
    stop_before = len(all_chunks)
  if ind == None:
     ind = range(start_at, stop_before)
  else:
     ind = ind[start_at:stop_before]
     
  chunks = [all_chunks[i] for i in ind if i < len(all_chunks)]  
  #Finally, return
  if return_ind:
    return chunks, ind
  else:
    return chunks

def works_by_author_qids(author_qids : list[str], limit_nr_qids_per_request = 300, time_out=5):
    if len(author_qids) < limit_nr_qids_per_request:
       author_qids = [author_qids]
    else:
       author_qids = chunker_alt(seq=author_qids, size=limit_nr_qids_per_request)

    works_by_authors = {}

    for author_qids_part in author_qids: 
      # QUERY HERE
      works_raw = query_works(author_qids_part)
      vars = works_raw['head']['vars']

      for n, work in enumerate(works_raw['results']['bindings']):
          work_gist = {}
          for label in vars:
              if label in work:
                  if ('language' == label) or ('qid' in label) and not ('Label' in label):
                      work_gist[label] = qid_from_url(work[label]['value'])
                  else:
                      work_gist[label] = work[label]['value']
              else:
                  print(f'{label} not in item nr {n}: {work}')
          if 'author_qid' in work_gist:
              author_qid = work_gist['author_qid']
              if author_qid not in works_by_authors:
                  works_by_authors[author_qid] = []
              works_by_authors[author_qid].append(work_gist)
      sleep(time_out)
    return works_by_authors

In [ ]:
chunker(range(6), size=4)

### Query all works by all our authors 

This takes about 3m.19s

In [ ]:
works_by_authors = works_by_author_qids(list(author_qids))
works_by_authors

In [ ]:
authors = pd.read_csv(Path(collections_dir, 'de_fiction_authors_metadata-2025-11-15T21:07.csv'), index_col=0)
authors = authors.assign(no_works=np.nan).astype({'no_works':object})
authors = authors.reset_index()
authors.index = authors['author']
authors = authors.drop(columns=['author'])
authors

In [ ]:
def no_case_punct(s : str):
    return re.sub(r'[^\w\s]', '', s).lower()


class MatchType(Enum):
    NO_MATCH = 0
    EXACT = 20
    NO_CASE_PUNCT = 19
    SUBSTR_MIDDLE = 8
    SUBSTR_END = 9
    SUBSTR_START = 10



    def __lt__(self, m : MatchType):
        return self.value < m.value
    
    def __float__(self):             # I thought this was for np.isna()
        return float(self.value)     # Well, if it was, that doesn't work


@dataclass
class MatchRecord:
    index : int         #index in our table
    type : MatchType
    wd_qid : str = ''
    field : str =''
    score : float = -1

# MAKE THIS GIANT LOOP A FUNCTION

matches = {}

metadata_authors_with_qid = metadata[metadata['author_qid'].notna()]

for i, author_qid, author_name, title, pubyear, work_qid in zip(metadata_authors_with_qid.index, 
                                                                metadata_authors_with_qid['author_qid'], 
                                                             metadata_authors_with_qid['author'], 
                                                metadata_authors_with_qid['title'], 
                                                metadata_authors_with_qid['year'], 
                                                metadata_authors_with_qid['work_qid']):
    if (author_qid in works_by_authors):
        #print(title)
        #print(author_qid)
        work_matches = []
        for work in works_by_authors[author_qid]:
            for field in ['title_de', 'title', 'work_qidLabel']:
                match_type = MatchType.NO_MATCH
                if (field in work):
                    #print(f'{title} ?= {work[field]}')
                    work_field= work[field]
                    if title == work_field:
                        msg = f'Exact match to {field}: {i} {title} by {author_name} with {work_qid}'
                        match_type = MatchType.EXACT
                        break # if this work matches our row, no need to look at other fields
                    
                    title_ncnp = no_case_punct(title)
                    work_field_ncnp = no_case_punct(work_field)
                    if title_ncnp == work_field_ncnp:
                        msg = f'No case/punct to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        match_type = MatchType.NO_CASE_PUNCT
                        break # if this work matches our row, no need to look at other fields
                    # Check for matches at beginning 
                    if re.match('^' + title_ncnp + '.*', work_field_ncnp) or re.match('^' + work_field_ncnp + '.*', title_ncnp):
                        msg = f'String match at start to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        logger.info(msg)
                        match_type = MatchType.SUBSTR_START
                        # This time, I don't think we should break
                    elif re.match('.*' + title_ncnp + '$', work_field_ncnp) or re.match('.*' + work_field_ncnp + '$', title_ncnp):
                        msg = f'String match at end to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        logger.info(msg)
                        match_type = MatchType.SUBSTR_START
                        # This time, I don't think we should break
                    elif (title_ncnp in work_field) or (work_field_ncnp in title):
                        msg = f'Substring match in middle to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        logger.info(msg)
                        match_type = MatchType.SUBSTR_MIDDLE
                    # SHOULD COMPUTE A FUZZY MATCHING METRIC HERE
            if match_type != MatchType.NO_MATCH:
                mrec = MatchRecord(index = i, wd_qid = work['work_qid'], type=match_type, field=field)
                print(msg)
            else:
                mrec = MatchRecord(index = i, type = match_type)
            work_matches.append(mrec)
        matches[i] = work_matches
                    
    else:
        # What does this actually mean: We have a qid for the author, so they exist on wikidata
        # But they do not seem to have any works associated to them
        # Worth investigating who they are!
        msg = f'Potential author without literary works:  {author_qid} ({author_name})'
        print(msg)
        logger.info(msg)
        authors.at[author_name, 'no_works'] = True
        # No need to repeat for this author!
        continue

In [ ]:
no_works_authors = authors[authors['no_works'].notna() & (authors['no_works'].notna() == True)].sort_values(['author_gender', 'author_last'])
no_works_authors

In [ ]:
summary = no_works_authors.drop(columns=['index', 'author_first', 'author_last', 'sources']).groupby('author_gender').sum()
summary['num_authors'] = no_works_authors.reset_index().groupby('author_gender')['author'].nunique()
summary = summary[['num_authors', 'author_qids', 
                   'works', 'work_qids', 'num_tokens', 
                   'num_sents', 'urls']].rename(columns={'num_tokens' : 'tokens', 'num_sents' : 'sents'})
summary

In [ ]:
author_qids_f_maybe_no_works = no_works_authors[no_works_authors['author_gender'] == 'f']['author_qid']
author_qids_f_maybe_no_works

In [ ]:
author_qids_maybe_no_works = no_works_authors['author_qid']
author_qids_maybe_no_works

The following verifies that these authors really all have no works.

In [ ]:
query_f = '''SELECT ?author_qid ?author_qidLabel ?sex_or_genderLabel (COUNT(?work) AS ?work_count)
WHERE {
  VALUES ?author_qid {
  wd:''' + " wd:".join(author_qids_f_maybe_no_works.values) + '''
  }
  # gender
  OPTIONAL { ?author_qid wdt:P21 ?sex_or_gender . }
  # works 
  OPTIONAL {
    ?work wdt:P50 ?author_qid .
    ?work wdt:P31 wd:Q7725634 .
  }
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en" .
  }
}
GROUP BY ?author_qid ?author_qidLabel ?sex_or_genderLabel
ORDER BY ?sex_or_genderLabel ?author_qidLabel ?author_qid'''

query_all = '''SELECT ?author_qid ?author_qidLabel ?sex_or_genderLabel (COUNT(?work) AS ?work_count)
WHERE {
  VALUES ?author_qid {
  wd:''' + " wd:".join(author_qids_maybe_no_works.values) + '''
  }
  # gender
  OPTIONAL { ?author_qid wdt:P21 ?sex_or_gender . }
  # works 
  OPTIONAL {
    ?work wdt:P50 ?author_qid .
    ?work wdt:P31 wd:Q7725634 .
  }
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en" .
  }
}
GROUP BY ?author_qid ?author_qidLabel ?sex_or_genderLabel
ORDER BY ?sex_or_genderLabel ?author_qidLabel ?author_qid'''

In [ ]:
result_f = wbi_helpers.execute_sparql_query(query_f, 
                                          user_agent=wbi_config['USER_AGENT'])
result_f

In [ ]:
result_all = wbi_helpers.execute_sparql_query(query_all, 
                                          user_agent=wbi_config['USER_AGENT'])
print(len(result_all['results']['bindings']))
result_all

In [ ]:
has_works_f = [record['author_qid']['value'] for record in result_f['results']['bindings'] 
             if record['work_count']['value'] != '0']
has_works_f

In [ ]:
has_works_all = [record['author_qid']['value'] for record in result_all['results']['bindings'] 
             if record['work_count']['value'] != '0']
has_works_all

In [ ]:
author_qids_no_works_f = [author_qid for author_qid in author_qids_f_maybe_no_works 
               if not author_qid in has_works_f]
len(author_qids_no_works_f)

In [ ]:
author_qids_no_works_all = [author_qid for author_qid in author_qids_maybe_no_works 
               if not author_qid in has_works_all]
len(author_qids_no_works_all)

# We must verify we don't have any works with multiple authors:

In [ ]:
metadata[metadata['author_qid'].isin(author_qids_no_works_all)]

In [ ]:
grouped = metadata[metadata['author_qid'].isin(author_qids_no_works_all)].groupby('title')
by_title = grouped['author'].nunique()
print(by_title.max())
by_title

In [ ]:
titles_several_authors = by_title[by_title != 1]
titles_several_authors

## Deal with parts of works

In [ ]:
view = metadata[metadata['author_qid'].isin(author_qids_no_works_all) \
                & ~metadata['title'].isin(titles_several_authors) \
                & metadata['title'].notna()]

select_part_of_works = view['title'].str.contains('Band', flags=re.IGNORECASE) \
    | view['title'].str.contains('Teil', flags=re.IGNORECASE) \
     | view['title'].str.contains('Theil', flags=re.IGNORECASE) \
        | view['title'].str.contains('Buch', flags=re.IGNORECASE) \
            | view['title'].str.contains('1', flags=re.IGNORECASE) \
            | view['title'].str.contains('2', flags=re.IGNORECASE) \
            | view['title'].str.contains('3 ', flags=re.IGNORECASE) \
            | view['title'].str.endswith(' I') \
            | view['title'].str.contains(' II', flags=re.IGNORECASE) \
            | view['title'].str.contains(' III', flags=re.IGNORECASE) \
            | view['title'].str.contains(' IV', flags=re.IGNORECASE) \
            | view['title'].str.endswith(' V') \
            | view['title'].str.endswith(' VI') \
            | view['title'].str.contains(' V ', flags=re.IGNORECASE) \
            | view['title'].str.contains(' VI ', flags=re.IGNORECASE) \
            | view['title'].str.contains(' VII', flags=re.IGNORECASE) \
            | view['title'].str.contains(' VIII', flags=re.IGNORECASE) \
            | view['title'].str.contains('Märchen', flags=re.IGNORECASE) \
            | view['title'].str.contains('Geschichten', flags=re.IGNORECASE) \
            | view['title'].str.contains('Novellen', flags=re.IGNORECASE) \
            | view['title'].str.contains('erste Reihe', flags=re.IGNORECASE) \
            | view['title'].str.contains('zweite Reihe', flags=re.IGNORECASE) \
            | view['title'].str.contains('dritte Reihe', flags=re.IGNORECASE) \
            | view['title'].str.contains('vierte Reihe', flags=re.IGNORECASE) \
            | view['title'].str.contains('Erzählungen', flags=re.IGNORECASE) \
            | view['title'].str.contains('/', flags=re.IGNORECASE) \
            | view['title'].str.contains('Roman', flags=re.IGNORECASE)

view[select_part_of_works]

In [ ]:
exclude = """Schuld und Strafe
Der Falschmünzer / Bestraft
Lemkes sel. Wwe.
Lemkes sel. Wwe. Das falsche Gebiß - Der blaue Amtsrichter - Berlin WW
Sprüche
Glossen
Em Hag no
Die Berliner Range VIII - Berlin wie es lebt und liebt
Die Berliner Range VII - Prosit Brautpaar!
Liebesnovellen der italienischen Renaissance
Gedichte in allerlei Humoren
Die Geschichte von Herrn Steinhausers Uhr
Satiren
Der »Stern von Afrika«
Lustige Gymnasialgeschichten
Ein Adjutantenritt und andere Militärhumoresken
Il Pantegan
Faraulip. Liebeslegenden aus der Südsee
Dorette lächelt ...
Saly's Revolutionstage
Lustiges aus dem Hundeleben und andere heitere Rundfunk-Vorträge
Neue Korfu-Geschichten
Ostseemärchen
Eine stille Welt - Novellen
Seegeschichten. Neue Folge
Seegeschichten. Zweite Sammlung
Seegeschichten
Grillparzers Liebesroman. Die Schwestern Fröhlich
Das Birken-Gräflein/Muckerl, der Taubennarr. Zwei Dorfgeschichten
Neue Geschichten aus dem Böhmerwald
Weißdornblüten aus dem Böhmerwälder und Wiener Volksleben
Johannes Volkh - Hausmittel der Liebe - Ein guter Mensch
Griechische Mythologie Theogonie, Götter
Humor (erste Reihe)
Gesammelte Erzählungen und Novellen. Fünfter und sechster
Daniel Junt / Die Himmelspacher
IrmelaEine Geschichte aus alter Zeit
Heil dir im Siegerkranz!: Erzählung(Zweite Auflage)
Bruder Leichtfuß und Stein am Bein :  roman
Der Bruderhof :  Eine bäuerliche Liebes- und Leidens-Geschichte"""

exclude = exclude.split('\n')
exclude

In [ ]:
print('\n'.join([title for title in view.loc[~select_part_of_works, 'title'] if title not in exclude]))

In [ ]:
print('\n'.join(set([title for title in view.loc[select_part_of_works, 'title']] + exclude)))

In [ ]:
view[~select_part_of_works & (~ view['title'].isin(exclude))]

In [ ]:
view[view['title'].str.contains("Bruderhof")].values

# Upload them!

### This is how you log in with OAuth
I currently cannot get this to work because the site won't allow me to validate my email address with the bot account, why I do not know.

In [ ]:
login_instance = wbi_login.OAuth2(consumer_token=CONSUMER_TOKEN, 
                                  consumer_secret=CONSUMER_SECRET, 
                                  mediawiki_api_url=wbi_config['MEDIAWIKI_API_URL'])
wbi = WikibaseIntegrator(login=login_instance, mediawiki_api=wbi_config['MEDIAWIKI_API_URL'])

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'



### Log in with a bot password (less preferred, apparently)

In [ ]:
#USER_AUTH = authorization['BOT_USERNAME']
#PASSWORD = authorization['BOT_USER_PWD']

USER_AUTH = authorization['BOT_AUTH']
PASSWORD = authorization['BOT_PWD']

login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD, mediawiki_api_url=wbi_config['MEDIAWIKI_API_URL'])

#login_instance = wbi_login.Login(user=authorization['BOT_USERNAME'], password=authorization['BOT_USER_PWD']) #, 
#                                 #user_agent=wbi_config['USER_AGENT'], 
#                                 #mediawiki_api_url=wbi_config['MEDIAWIKI_API_URL'])

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'


## This is where we finally upload

In [ ]:
#view = metadata[metadata['author_qid'].isin(author_qids_no_works) & (~select_part_of_works)]
#view

sel_for_upload = view[~select_part_of_works & (~ view['title'].isin(exclude))][1:2]
sel_for_upload

In [ ]:
bot = Bot(wbi,login_instance, EDIT_SUMMARY,logger, save_dir=collections_dir)

bot.upload(sel_for_upload, metadata, dry_run = False)